In [1]:
import os
# Force the system to only see GPU #1. (Change to 0, 2, 3 etc. as needed)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
from datasets import load_dataset

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # normalized float 4 — better accuracy than plain int4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # extra quantization of the quantization constants, saves a bit more memory
)

In [4]:
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-32B-Thinking",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 2/1058 [00:00<07:11,  2.44it/s]/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 1058/1058 [00:18<00:00, 56.64it/s] 


In [5]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-32B-Thinking")

### Testing Quilt-VQA

In [6]:
quilt_vqa_dataset = load_dataset('wisdomik/Quilt_VQA')

In [7]:
quilt_vqa_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer', 'answer_type', 'context'],
        num_rows: 985
    })
})

### Single Instance Testing

In [9]:
data_idx = 0
data_inst = quilt_vqa_dataset['train'][data_idx]
data_inst

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1920x1080>,
 'question': 'What is the visual characteristic of the cells after therapy?',
 'answer': 'After therapy, the cells are following the path of neuroendocrine differentiation and forming ganglion cells with big prominent cell nuclei and abundant eosinophilic cytoplasm.',
 'answer_type': 'OPEN',
 'context': " This is differentiation following therapy, and of course, this is the ultimate form of differentiation following therapy. It's following that path of neuroendocrine differentiation and this time around forming ganglion cells with those big prominent cell nuclei and that abundant eosinophilic cytoplasm. Isn't that pretty? If there was only one marker that you were allowed to use to make a diagnosis of Ewing's, it would be CD99. What you're looking for is diffuse, virtually all of the cells should be positive, positivity for CD99."}

In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": data_inst["image"]},
            {"type": "text", "text": data_inst['question']},
        ],
    }
]

In [12]:
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [14]:
image_inputs, video_inputs = process_vision_info(messages)

In [18]:
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(model.device)

In [22]:
generated_ids = model.generate(**inputs, max_new_tokens=1024)

/data/mn27889/miniconda3/envs/path-opendata-vlms/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [24]:
output_text = processor.batch_decode(
    generated_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)

In [27]:
print(output_text[0])

So, let's look at the image. The title says "Ewing sarcoma with ganglionic differentiation post chemotherapy." First, I need to recall what post-chemotherapy changes might look like in a histopathology image. Chemotherapy often causes changes in tumor cells, like necrosis, reduced cellularity, or differentiation.

In this image, the cells have a mix of features. The ganglionic differentiation part means some cells might resemble ganglion cells, which have large nuclei, maybe with a nucleolus, and a more mature appearance. After chemo, the tumor might show less aggressive features or signs of treatment effect.

Looking at the cells: there are cells with large, round to oval nuclei, some with prominent nucleoli (which is typical of ganglionic differentiation—ganglion cells have large nuclei with nucleoli). The background stroma is pink (H&E stain, so cytoplasm and connective tissue), and there's a mix of cell types. Post-chemotherapy, we might see reduced mitotic activity, more mature ce

### Batch Testing

In [33]:
batch_size = 4
batch_data = quilt_vqa_dataset['train'][0:batch_size]
batch_images = [item.convert("RGB") for item in batch_data['image']]
batch_prompts = [item for item in batch_data['question']]

In [35]:
messages_list = [
    [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ],
    }]
    for img, prompt in zip(batch_images, batch_prompts)
]

In [36]:
messages_list

[[{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=1920x1080>},
    {'type': 'text',
     'text': 'What is the visual characteristic of the cells after therapy?'}]}],
 [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=1920x1080>},
    {'type': 'text',
     'text': "What is the initial hint that the image does not depict Ewing's tumors?"}]}],
 [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=1920x1080>},
    {'type': 'text',
     'text': 'What are the signs of aberrant maturation, hyperkeratosis, and acanthosis in the image?'}]}],
 [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=1920x1080>},
    {'type': 'text',
     'text': 'Do we see hyperchromasia and enlargement in the image?'}]}]]

In [37]:
# Apply chat template to each
batch_texts = [
    processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
    for m in messages_list
]

In [38]:
batch_texts

['<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>What is the visual characteristic of the cells after therapy?<|im_end|>\n<|im_start|>assistant\n<think>\n',
 "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>What is the initial hint that the image does not depict Ewing's tumors?<|im_end|>\n<|im_start|>assistant\n<think>\n",
 '<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>What are the signs of aberrant maturation, hyperkeratosis, and acanthosis in the image?<|im_end|>\n<|im_start|>assistant\n<think>\n',
 '<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Do we see hyperchromasia and enlargement in the image?<|im_end|>\n<|im_start|>assistant\n<think>\n']

In [39]:
# Process the whole batch together
inputs = processor(
    text=batch_texts,
    images=batch_images,
    padding=True,
    return_tensors="pt",
).to(model.device)

In [40]:
# Generate
generated_ids = model.generate(**inputs, max_new_tokens=1024)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [41]:
# Decode
outputs = processor.batch_decode(
    generated_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True,
)

In [42]:
for prompt, output in zip(batch_prompts, outputs):
    print(f"Prompt: {prompt}\nOutput: {output}\n")

Prompt: What is the visual characteristic of the cells after therapy?
Output: Lo
So, let's look at the image. The title says "Ewing sarcoma with ganglionic differentiation post chemotherapy." So we need to describe the visual features. 

First, in histopathology, post-chemotherapy changes often show necrosis, reduced tumor cellularity, or specific differentiation. Here, ganglionic differentiation means the tumor cells are showing features of ganglion cells. Let's analyze the image: the cells have large, round to oval nuclei with prominent nucleoli, and the cytoplasm might be more abundant. Also, post-chemotherapy, there might be less mitotic activity, but the key here is ganglionic differentiation. 

Ganglionic differentiation in Ewing sarcoma would show cells with features like large, mature-appearing nuclei, possibly with Nissl substance (but in H&E, it's the nuclear and cytoplasmic morphology). The cells in the image have varied nuclear sizes, some with prominent nucleoli, and the c